<a href="https://colab.research.google.com/github/atharvac25/BERT/blob/initial-commit/distiledBert_WORKING_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets nltk scikit-learn

import os
import torch
import nltk
import json
import numpy as np
from nltk.tokenize import sent_tokenize
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, f1_score
nltk.download('punkt')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 16.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system 

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [2]:
!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=c82783bae1692b00332fc7e171e4b4e35ba3e6b04ec1f9e61fb60b69ae02d23b
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge-score


In [3]:
import json
import re
import nltk
import numpy as np
from nltk.tokenize import sent_tokenize
from transformers import BertTokenizer
from rouge_score import rouge_scorer

nltk.download('punkt')

# -------------------------
#  Load Extractive Articles from Subset
# -------------------------

def load_newsroom_extractive(path, limit=500):
    dataset = []
    count = 0

    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                obj = json.loads(line)
                if obj.get("density_bin") == "extractive":
                    dataset.append({
                        "article": obj["text"],
                        "highlights": obj["summary"]
                    })
                    count += 1
                if count >= limit:
                    break
            except Exception:
                continue  # skip bad lines

    print(f"✅ Loaded {len(dataset)} extractive articles")
    return dataset

# -------------------------
#  Sentence Tokenization + Cleaning
# -------------------------

def clean_article(text):
    text = text.replace("\n", " ")
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def safe_sent_tokenize(text):
    try:
        return sent_tokenize(text)
    except Exception:
        return re.split(r'(?<=[.!?]) +', text)

# -------------------------
#  Preprocessing Function
# -------------------------

def preprocess_newsroom(dataset, max_sentences=60, max_len=512, bert_model='distilbert-base-uncased', rouge_threshold=0.2):
    tokenizer = BertTokenizer.from_pretrained(bert_model)
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    processed_data = []

    for idx, item in enumerate(dataset):
        if idx % 100 == 0:
            print(f"🔁 Processing article {idx}/{len(dataset)}")

        article = clean_article(item['article'])
        summary = item['highlights']
        sentences = safe_sent_tokenize(article)[:max_sentences]

        if not sentences:
            continue

        tokenized = [tokenizer.encode(
            sent, add_special_tokens=True, max_length=max_len,
            truncation=True, padding='max_length'
        ) for sent in sentences]

        attention_masks = [[1 if t != tokenizer.pad_token_id else 0 for t in tokens] for tokens in tokenized]

        labels = []
        for sent in sentences:
            score = scorer.score(sent, summary)['rougeL'].fmeasure
            labels.append(1 if score >= rouge_threshold else 0)

        if sum(labels) == 0:
            top_idx = np.argmax([scorer.score(sent, summary)['rougeL'].fmeasure for sent in sentences])
            labels[top_idx] = 1

        processed_data.append({
            'input_ids': tokenized,
            'attention_mask': attention_masks,
            'labels': labels,
            'sentence_count': len(sentences)
        })

    avg_pos = sum(sum(ex['labels']) for ex in processed_data) / len(processed_data)
    print(f"\n🔢 Avg positive sentences per article: {avg_pos:.2f}")
    return processed_data

# -------------------------
#  Run the Full Pipeline
# ------------------------

# Step 1: Upload in Colab or ensure it's in your working dir
data_path = "train_subset.jsonl"
news_data = load_newsroom_extractive(data_path, limit=500)

# Step 2: Preprocess and label
processed_data = preprocess_newsroom(news_data)

# Step 3: View a sample
print("\n Sample processed item:")
sample = processed_data[0]
print(f"Sentence count: {sample['sentence_count']}")
print(f"Labels: {sample['labels']}")
print(f"First input_ids (truncated): {sample['input_ids'][0][:10]}...")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


✅ Loaded 500 extractive articles


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DistilBertTokenizer'. 
The class this function is called from is 'BertTokenizer'.


🔁 Processing article 0/500
🔁 Processing article 100/500
🔁 Processing article 200/500
🔁 Processing article 300/500
🔁 Processing article 400/500

🔢 Avg positive sentences per article: 3.03

 Sample processed item:
Sentence count: 37
Labels: [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0]
First input_ids (truncated): [101, 2065, 21100, 3550, 3237, 3477, 2038, 5262, 2468, 1037]...


In [4]:
from datasets import load_dataset

def unify(dataset, article_key, summary_key):
    return [{"article": item[article_key], "highlights": item[summary_key]} for item in dataset]

# ------------------ Load & Filter ------------------
print("Loading datasets...")

# PubMed
pubmed = load_dataset("scientific_papers", "pubmed", split="train[:500]")
pubmed_data = unify(pubmed, "article", "abstract")

# Multi-News
multinews = load_dataset("multi_news", split="train[:500]")
multinews_data = unify(multinews, "document", "summary")

# CNN/DailyMail
cnn = load_dataset("cnn_dailymail", "3.0.0", split="train[:500]")
cnn_data = unify(cnn, "article", "highlights")

# Newsroom — already uploaded & subsetted by you
import json
def load_newsroom_subset(path, limit=500):
    dataset = []
    with open(path, 'r') as f:
        for line in f:
            obj = json.loads(line)
            if obj.get("density_bin") == "extractive":
                dataset.append({
                    "article": obj["text"],
                    "highlights": obj["summary"]
                })
                if len(dataset) >= limit:
                    break
    return dataset

newsroom_data = load_newsroom_subset("train_subset.jsonl", limit=500)

# ------------------ Combine All ------------------
combined_data = newsroom_data + pubmed_data + multinews_data + cnn_data
print(f"Total combined articles: {len(combined_data)}")

# ------------------ Run Preprocessing ------------------
processed_combined = preprocess_newsroom(combined_data, rouge_threshold=0.2)

# ------------------ Analyze Per Dataset ------------------
names = ["Newsroom", "PubMed", "Multi-News", "CNN/DailyMail"]
for i, name in enumerate(names):
    chunk = processed_combined[i*500:(i+1)*500]
    avg = sum(sum(x['labels']) for x in chunk) / 500
    avg_len = sum(x['sentence_count'] for x in chunk) / 500
    print(f"\n {name}:")
    print(f"   ➤ Avg labeled sentences per article: {avg:.2f}")
    print(f"   ➤ Avg total sentences per article: {avg_len:.2f}")


Loading datasets...


README.md:   0%|          | 0.00/8.27k [00:00<?, ?B/s]

scientific_papers.py:   0%|          | 0.00/5.35k [00:00<?, ?B/s]

The repository for scientific_papers contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/scientific_papers.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/119924 [00:00<?, ? examples/s]

...

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DistilBertTokenizer'. 
The class this function is called from is 'BertTokenizer'.


Total combined articles: 2000
🔁 Processing article 0/2000
🔁 Processing article 100/2000
🔁 Processing article 200/2000
🔁 Processing article 300/2000
🔁 Processing article 400/2000
🔁 Processing article 500/2000
🔁 Processing article 600/2000
🔁 Processing article 700/2000
🔁 Processing article 800/2000
🔁 Processing article 900/2000
🔁 Processing article 1000/2000
🔁 Processing article 1100/2000
🔁 Processing article 1200/2000
🔁 Processing article 1300/2000
🔁 Processing article 1400/2000
🔁 Processing article 1500/2000
🔁 Processing article 1600/2000
🔁 Processing article 1700/2000
🔁 Processing article 1800/2000
🔁 Processing article 1900/2000

🔢 Avg positive sentences per article: 2.15

 Newsroom:
   ➤ Avg labeled sentences per article: 3.03
   ➤ Avg total sentences per article: 29.53

 PubMed:
   ➤ Avg labeled sentences per article: 1.66
   ➤ Avg total sentences per article: 53.07

 Multi-News:
   ➤ Avg labeled sentences per article: 1.25
   ➤ Avg total sentences per article: 48.14

 CNN/DailyMail

In [5]:
def preprocess_newsroom(
    dataset,
    max_sentences=120,
    max_len=512,
    bert_model='distilbert-base-uncased',
    rouge_threshold=0.2,
    min_total_sentences=30,
    min_positive_labels=3
):
    from transformers import DistilBertTokenizer
    tokenizer = DistilBertTokenizer.from_pretrained(bert_model)
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

    processed_data = []
    filtered_out = 0

    for idx, item in enumerate(dataset):
        if idx % 100 == 0:
            print(f"🔁 Processing article {idx}/{len(dataset)}")

        article = clean_article(item['article'])
        summary = item['highlights']
        sentences = safe_sent_tokenize(article)[:max_sentences]

        if len(sentences) < min_total_sentences:
            filtered_out += 1
            continue

        tokenized = [tokenizer.encode(
            sent, add_special_tokens=True, max_length=max_len,
            truncation=True, padding='max_length'
        ) for sent in sentences]

        attention_masks = [[1 if t != tokenizer.pad_token_id else 0 for t in tokens] for tokens in tokenized]

        labels = []
        for sent in sentences:
            score = scorer.score(sent, summary)['rougeL'].fmeasure
            labels.append(1 if score >= rouge_threshold else 0)

        if sum(labels) < min_positive_labels:
            filtered_out += 1
            continue

        processed_data.append({
            'input_ids': tokenized,
            'attention_mask': attention_masks,
            'labels': labels,
            'sentence_count': len(sentences)
        })

    avg_pos = sum(sum(ex['labels']) for ex in processed_data) / len(processed_data)
    print(f"\nFinal high-quality articles kept: {len(processed_data)}")
    print(f" Articles filtered out: {filtered_out}")
    print(f" Avg positive sentences per article: {avg_pos:.2f}")

    return processed_data


In [6]:
processed_filtered = preprocess_newsroom(
    combined_data,
    rouge_threshold=0.2,
    min_total_sentences=30,
    min_positive_labels=3
)

🔁 Processing article 0/2000
🔁 Processing article 100/2000
🔁 Processing article 200/2000
🔁 Processing article 300/2000
🔁 Processing article 400/2000
🔁 Processing article 500/2000
🔁 Processing article 600/2000
🔁 Processing article 700/2000
🔁 Processing article 800/2000
🔁 Processing article 900/2000
🔁 Processing article 1000/2000
🔁 Processing article 1100/2000
🔁 Processing article 1200/2000
🔁 Processing article 1300/2000
🔁 Processing article 1400/2000
🔁 Processing article 1500/2000
🔁 Processing article 1600/2000
🔁 Processing article 1700/2000
🔁 Processing article 1800/2000
🔁 Processing article 1900/2000

Final high-quality articles kept: 388
 Articles filtered out: 1612
 Avg positive sentences per article: 4.14


In [7]:
import json

def load_newsroom_subset(path, start, end):
    dataset = []
    with open(path, 'r') as f:
        for i, line in enumerate(f):
            if i < start:
                continue
            if i >= end:
                break
            obj = json.loads(line)
            if obj.get("density_bin") == "extractive":
                dataset.append({
                    "article": obj["text"],
                    "highlights": obj["summary"]
                })
    return dataset

# Loop to keep loading more chunks until we reach 1000 high-quality articles
high_quality_articles = []
start = 0
chunk_size = 1000
target_count = 1000

while len(high_quality_articles) < target_count:
    print(f"\n Loading chunk: {start} to {start + chunk_size}")
    batch = load_newsroom_subset("train_subset.jsonl", start, start + chunk_size)

    filtered = preprocess_newsroom(
        batch,
        rouge_threshold=0.2,
        min_total_sentences=30,
        min_positive_labels=3,
        max_sentences=100
    )

    high_quality_articles += filtered
    print(f"High-quality total so far: {len(high_quality_articles)}")

    start += chunk_size

    if not batch:  # end of file
        print("No more data to process.")
        break



 Loading chunk: 0 to 1000
🔁 Processing article 0/332
🔁 Processing article 100/332
🔁 Processing article 200/332
🔁 Processing article 300/332

Final high-quality articles kept: 112
 Articles filtered out: 220
 Avg positive sentences per article: 4.21
High-quality total so far: 112

 Loading chunk: 1000 to 2000
🔁 Processing article 0/327
🔁 Processing article 100/327
🔁 Processing article 200/327
🔁 Processing article 300/327

Final high-quality articles kept: 103
 Articles filtered out: 224
 Avg positive sentences per article: 4.39
High-quality total so far: 215

 Loading chunk: 2000 to 3000
🔁 Processing article 0/356
🔁 Processing article 100/356
🔁 Processing article 200/356
🔁 Processing article 300/356

Final high-quality articles kept: 127
 Articles filtered out: 229
 Avg positive sentences per article: 4.12
High-quality total so far: 342

 Loading chunk: 3000 to 4000
🔁 Processing article 0/332
🔁 Processing article 100/332
🔁 Processing article 200/332
🔁 Processing article 300/332

Final 

In [8]:
# 💾 Save the top 1000 filtered examples
with open("final_filtered_1000.json", "w") as f:
    json.dump(high_quality_articles[:1013], f)


In [9]:
!pip install transformers datasets scikit-learn


In [10]:
import json
from sklearn.model_selection import train_test_split

# Load dataset
with open("final_filtered_1000.json", "r") as f:
    data = json.load(f)

# Flatten sentence-level examples
examples = []
for article in data:
    for input_ids, mask, label in zip(article['input_ids'], article['attention_mask'], article['labels']):
        examples.append({
            'input_ids': input_ids,
            'attention_mask': mask,
            'label': label
        })

# Train/test split
train_data, test_data = train_test_split(examples, test_size=0.2, random_state=42)

In [11]:
!pip install -U transformers
!pip install "huggingface_hub[hf_xet]"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 MB 12.6 MB/s eta 0:00:00


In [12]:
!pip install -U transformers huggingface_hub


In [13]:
!pip install -U transformers

In [14]:
import json
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset

# Load flattened dataset from JSON
with open("final_filtered_1000.json", "r") as f:
    data = json.load(f)

# Flatten sentence-level examples
examples = []
for article in data:
    for input_ids, mask, label in zip(article['input_ids'], article['attention_mask'], article['labels']):
        examples.append({
            'input_ids': input_ids,
            'attention_mask': mask,
            'label': label
        })

# Train/test split
train_data, test_data = train_test_split(examples, test_size=0.2, random_state=42)

# Custom Dataset class
class ExtractiveDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': torch.tensor(item['input_ids']),
            'attention_mask': torch.tensor(item['attention_mask']),
            'labels': torch.tensor(item['label'])
        }

train_dataset = ExtractiveDataset(train_data)
test_dataset = ExtractiveDataset(test_data)

In [16]:
from transformers import DistilBertForSequenceClassification, Trainer, TrainingArguments

import os
os.environ["WANDB_DISABLED"] = "true"  # disables W&B logging


model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",  # Changed from evaluation_strategy
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"  # Changed to match compute_metrics
)

def compute_metrics(p):
    from sklearn.metrics import accuracy_score, f1_score
    preds = p.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(p.label_ids, preds),
        "f1": f1_score(p.label_ids, preds)
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.268500,0.259180,0.916867,0.109560
2,0.190100,0.286716,0.908945,0.235690
3,0.181700,0.350688,0.906137,0.243942


TrainOutput(global_step=7479, training_loss=0.21700516593760708, metrics={'train_runtime': 5969.1328, 'train_samples_per_second': 20.046, 'train_steps_per_second': 1.253, 'total_flos': 1.585038658618368e+16, 'train_loss': 0.21700516593760708, 'epoch': 3.0})

In [ ]:
!zip -r distilled_summary_model.zip results final_filtered_1000.json train_subset.jsonl


  adding: results/ (stored 0%)
  adding: results/checkpoint-2493/ (stored 0%)
  adding: results/checkpoint-2493/rng_state.pth (deflated 25%)
  adding: results/checkpoint-2493/training_args.bin (deflated 51%)
  adding: results/checkpoint-2493/optimizer.pt (deflated 17%)
  adding: results/checkpoint-2493/model.safetensors (deflated 8%)
  adding: results/checkpoint-2493/trainer_state.json (deflated 74%)
  adding: results/checkpoint-2493/config.json (deflated 45%)
  adding: results/checkpoint-2493/scheduler.pt (deflated 56%)
  adding: results/checkpoint-7479/ (stored 0%)
  adding: results/checkpoint-7479/rng_state.pth (deflated 25%)
  adding: results/checkpoint-7479/training_args.bin (deflated 51%)
  adding: results/checkpoint-7479/optimizer.pt (deflated 17%)
  adding: results/checkpoint-7479/model.safetensors (deflated 8%)
  adding: results/checkpoint-7479/trainer_state.json (deflated 77%)
  adding: results/checkpoint-7479/config.json (deflated 45%)
  adding: results/checkpoint-7479/sched

In [ ]:
from google.colab import files
files.download("distilled_summary_model.zip")


In [ ]:
model


In [ ]:
import torch
import nltk
from nltk.tokenize import sent_tokenize
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

nltk.download('punkt')

# 🧠 Load tokenizer if needed (skip if already loaded)
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def top_k_summary(article_text, model, tokenizer, k=3, debug=True):
    sentences = sent_tokenize(article_text)
    if not sentences:
        print("No valid sentences.")
        return []

    inputs = tokenizer(sentences, padding=True, truncation=True, max_length=512, return_tensors="pt")

    # Move tensors to same device as model
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        scores = probs[:, 1].tolist()

    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    summary = [sentences[i] for i in top_indices]

    if debug:
        print("🧪 Sentence Probabilities:")
        for i, (sent, score) in enumerate(zip(sentences, scores)):
            print(f"{score:.4f} ➤ {sent}")

    print("\n🧠 Top-k Extractive Summary:")
    for sent in summary:
        print("–", sent)

    return summary

sample_text = """
Apple Inc. unveiled its latest iPhone model during a virtual event held on Tuesday, introducing several new features aimed at enhancing user experience.
The iPhone 15 now includes a periscope-style telephoto lens, allowing users to capture high-quality zoomed-in photos.
Apple also announced an upgraded A17 chip that offers improved performance and battery life.
The event, streamed live on Apple’s website, was watched by over two million people.
CEO Tim Cook emphasized Apple’s commitment to innovation and environmental responsibility.
The new iPhone will ship in four colors and starts at $799, with pre-orders beginning this Friday.
Analysts predict strong sales driven by both loyal customers and emerging markets.
Some critics, however, pointed out that the changes were incremental compared to previous releases.
"""

top_k_summary(sample_text, model, tokenizer, k=3, debug=True)


In [ ]:
!pip install newspaper3k
!pip install openai python-dotenv


In [ ]:
import torch
from newspaper import Article
from nltk.tokenize import sent_tokenize
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
import openai
import os
from dotenv import load_dotenv


In [ ]:
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")


In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
model = DistilBertForSequenceClassification.from_pretrained("./results/checkpoint-4986")  # or your latest checkpoint
model.eval()


In [ ]:
def scrape_article(url):
    article = Article(url)
    article.download()
    article.parse()
    return article.text


In [ ]:
def summarize_extractive(text, threshold=0.3):
    sentences = sent_tokenize(text)
    inputs = tokenizer(sentences, truncation=True, padding=True, max_length=512, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        scores = probs[:, 1].tolist()

    summary = [s for s, p in zip(sentences, scores) if p > threshold]
    return summary


In [ ]:
def generate_abstractive_summary(sentences):
    prompt = (
        "Rewrite the following sentences into a fluent paragraph preserving all facts:\n\n"
        + "\n".join(sentences)
    )
    response = openai.ChatCompletion.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response['choices'][0]['message']['content'].strip()


In [ ]:
url = "https://www.bbc.com/news/technology-66796338"  # replace with any article URL
text = scrape_article(url)

print("Article scraped. Generating extractive summary...")
extractive = summarize_extractive(text)

print("\nExtractive Summary:")
for s in extractive:
    print("-", s)

abstractive = generate_abstractive_summary(extractive)

print("\nAbstractive (GPT) Summary:\n", abstractive)
